In [1]:
# ALL FUNCTIONS

import os, re, pickle
import pandas as pd
import numpy as np
from itertools import chain

def _remove_strings(text):
    #remove common strings from test. 
    #patterns = ['HEPG2_', 'SKNSH_', 'GM12878_', 'K562_', 'MCF7_','_HEPG2', r'\.summary\.txt', r'\.mat', r'-cellFilt']
    patterns = [r'\.summary\.txt', r'\.mat', r'-cellFilt', r'\_metrics.csv']
    for pattern in patterns:
        text = re.sub(pattern, '', text)
    return text
    
    
def _make_contingency_table(df_expected, df_observed): 
    contingency_table = {
            'TP': np.sum(np.minimum(df_observed, df_expected), axis=0),      # Minimum of the two values is true positive
            'FP': np.sum(np.maximum(0, df_observed - df_expected), axis=0),  # Difference, where df_observed > df_expected
            'FN': np.sum(np.maximum(0, df_expected - df_observed), axis=0),  # Difference, where df_expected > df_observed
            'TN': np.sum((df_observed == 0) & (df_expected == 0), axis=0)
        }
    contingency_df = pd.DataFrame(contingency_table)
    return(contingency_df)

    
    
def _calc_scores(df, averaging_type):
    
    if averaging_type=="micro":
        #micro = sum all TPs and then calculate F1
        #will return three values, 
        
        tp = df["TP"].sum()
        fp = df["FP"].sum()
        fn = df["FN"].sum()
        if tp==0 and fp==0 and fn==0: 
            precision = 0; recall=0; f1=0
        else:  
            precision = (tp/(tp+fp))
            recall = (tp/ (tp+fn))
            f1 = ( tp/ (tp + (0.5*(fp+fn)) ) )
            
        return pd.Series([precision, recall, f1], index=["precision","recall","f1"])

    elif averaging_type=="macro":
        #macro = calculate F1s and then take average F1. 
        #will return three lists, 
        
        df['precision'] = np.where(df['TP'] + df['FP'] == 0, np.nan, 
                                   df['TP'] / (df['TP'] + df['FP']))
        df['recall']    = np.where(df['TP'] + df['FN'] == 0, np.nan, 
                                   df['TP'] / (df['TP'] + df['FN']))
        df['f1']        = np.where(df['TP'] + (0.5 * (df['FP'] + df['FN'])) == 0, np.nan,
                                   df['TP'] / (df['TP'] + (0.5 * (df['FP'] + df['FN']))))
        return(df)
        
        
        
def _get_missing(mat_dir, metric_dir, analysis_type, cl):
    
    #check if the metric files already exist given the mats. 
    #if this is a clean directory it will just be everything. 
    has_mats = [re.sub('-cellFilt|.mat','', x) for x in os.listdir(mat_dir) if x.endswith(".mat") and cl in x]
    has_metrics = [re.sub('-cellFilt|_metrics.csv','', x) for x in os.listdir(metric_dir) if x.endswith("_metrics.csv") and cl in x]

    filtered_metrics = []
    for f in has_metrics:
        done = np.unique(pd.read_csv(f'{metric_dir}/{f}_metrics.csv', usecols=["type"]).type.values)
        if analysis_type in done: 
            filtered_metrics.append(f)
    
    missing = list(set(has_mats).difference(set(filtered_metrics)))
    files = [f"{x}-cellFilt" if x == cl else x for x in missing]
    files = [f"{x}.mat" for x in files]
    return files


    
def CalculateMetrics(mat_dir, metric_dir, cell_line, keep_peaks=None, analysis_type="tf"):
    try:
        files = _get_missing(mat_dir, metric_dir, analysis_type, cell_line)
        print(f"{cell_line}: need to add information for {len(files)} files")
        if len(files) == 0: return 1
        
        #get the baseline dataframe, and summarize it based on type. 
        df_expected = pd.read_csv(f"{mat_dir}/{cell_line}-cellFilt.mat", index_col=0)
        if keep_peaks is not None:  df_expected = df_expected.loc[keep_peaks]

        if analysis_type == "peak":      df_expected_peak = df_expected.sum(axis=1).to_frame(name='sum')
        elif analysis_type == "tf":      True
        elif analysis_type == "class":   df_expected_class, tfs_in_unit = byClass(df_expected)
        elif analysis_type == "cluster": df_expected_cluster, tfs_in_unit = byCluster(df_expected)
        else: print("internal error in calculte metrics"); return

        for f in files:
            cond_name = _remove_strings(f)
            print(f"...{cond_name}")
            df_observed = pd.read_csv(f"{mat_dir}/{f}", index_col=0)
            if keep_peaks is not None: df_observed = df_observed.loc[keep_peaks] 

            if analysis_type == "peak":
                series_o = df_observed.sum(axis=1)
                cont_df = _make_contingency_table(df_expected_peak, series_o.to_frame(name='sum'))

            elif analysis_type == "tf":
                cont_df = _make_contingency_table(df_expected, df_observed)

            elif analysis_type == "class":
                df_observed_class = byClass(df_observed, tfs_in_unit)
                cont_df = _make_contingency_table(df_expected_class, df_observed_class)

            elif analysis_type == "cluster":
                df_observed_cluster = byCluster(df_observed, tfs_in_unit)
                cont_df = _make_contingency_table(df_expected_cluster, df_observed_cluster)

            else:
                print("internal error in calculte metrics"); return


            cont_df_with_metrics = _calc_scores(cont_df, averaging_type="macro")
            cont_df_with_metrics["type"] = analysis_type

            outfile = f"{metric_dir}/{cond_name}_metrics.csv"
            if not os.path.isfile(outfile):
                cont_df_with_metrics.to_csv(outfile)
            else: 
                print("appending results to existing file") #so you can stack tf, cluster, peak, etc. 
                cont_df_with_metrics.to_csv(outfile, mode='a', header=False)
            
        return 1
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None 
    
    
def CalculateGroupMetrics(data_dir, cl, analysis_type="tf"):
    files = [x for x in os.listdir(data_dir) if x.endswith("_metrics.csv") and cl in x]
    
    byGroup_df = pd.DataFrame() 

    conds = [i.split('_')[1] for i in files]
    conds.remove('metrics.csv')
    
    for group in np.unique(conds):
        p_list = []; r_list = []; f_list=[]
        group_files = [i for i in files if f"_{group}_" in i]

        joint = pd.DataFrame()
        for gf in group_files:
            df = pd.read_csv(f"{data_dir}/{gf}", index_col=0)
            df = df.loc[df["type"] == analysis_type]
            joint = pd.concat([joint, df], axis=0)

        micro_df = _calc_scores(joint, averaging_type="micro")
        mean_df = joint[["precision", "recall", "f1"]].apply(lambda col: col.mean())
        std_df  = joint[["precision", "recall", "f1"]].apply(lambda col: col.std())
        byGroup_df[f"{cl}_{group}"] = pd.concat([micro_df.add_suffix('_micro'),
                                          mean_df.add_suffix('_mean'), 
                                          std_df.add_suffix('_sd')], axis=0)
    return(byGroup_df.T)



def AddSamplingStats(df, row_type="sample"):
    
    def transform_to_dict(file_list):
        result = {}
        for item in file_list:
            prefix = "_".join(item.split('_')[:-1])
            if prefix not in result:
                result[prefix] = []
            result[prefix].append(item)
        return result
    
    all_metrics = pd.read_csv("../04_downsampling/sampling_stats.csv", sep=",", index_col=0)
    all_metrics.index = [re.sub("_dict" , "", x) for x in all_metrics.index]
    
    if row_type=="sample":
        df_INFO = df.merge(all_metrics, right_index=True, left_index=True, how='left')
    
    elif row_type=="group":
        grouping_dict = transform_to_dict(all_metrics.index)
        rows = []
        
        new_df = pd.DataFrame() 
        for g in grouping_dict.keys():
            subf = grouping_dict[g]
            group_df = all_metrics.loc[subf]
            group_df = group_df.drop(columns=["seed"])
            str_columns = ["cell_line", "sampling_type", "value"]
            str_series  = group_df[str_columns].iloc[0]
            mean_series = group_df.drop(columns=str_columns).mean(axis=0)

            combined_series = pd.concat([str_series, mean_series])
            combined_series["group"] = g
            rows.append(combined_series)
            
        new_df = pd.DataFrame(rows)
        new_df.index = new_df.group
        new_df.drop(columns=["group"], inplace=True)
        
        df_INFO = df.merge(new_df, right_index=True, left_index=True, how='left')
    
    return(df_INFO)


def JoinMetrics(metric_dir, cl, analysis_type):
    files = [x for x in os.listdir(metric_dir) if x.endswith("_metrics.csv") and cl in x]
    
    joint = pd.DataFrame()
    for infile in files:
        cond_name = _remove_strings(infile)

        byTF_df = pd.read_csv(f"{metric_dir}/{infile}", index_col=0)
        byTF_df = byTF_df.loc[byTF_df["type"] == analysis_type]

        micro_df = _calc_scores(byTF_df, averaging_type="micro")
        mean_df = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.mean())
        std_df  = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.std())
        joint[cond_name] = pd.concat([micro_df.add_suffix('_micro'),
                                      mean_df.add_suffix('_mean'), 
                                      std_df.add_suffix('_sd')], axis=0)
    
    return joint.T


## All Peaks

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
from pathlib import Path

# t = "hint" 
# data_dir = f"../06_dataquality/{t}/" #will look for mats
# mat_dir = f"{data_dir}/mats/"
# metric_dir = f"{data_dir}/metrics/no_threshold/"

# t = "tobias" 
# data_dir = f"../06_dataquality/{t}/" #will look for mats
# mat_dir = f"{data_dir}/mats/static_threshold/"
# metric_dir = f"{data_dir}/metrics/static_threshold/"

t = "print" 
data_dir = f"../06_dataquality/{t}/" 
mat_dir = f"{data_dir}/mats/thresh_03/"
metric_dir = f"{data_dir}/metrics/thresh_03/"

Path(metric_dir).mkdir(parents=True, exist_ok=True)

for cl in ["K562", "HEPG2", "GM12878", "MCF7", "SKNSH"]:
    for analysis_type in ["peak", "tf"]:
        print(analysis_type)
        
        val = CalculateMetrics(mat_dir=mat_dir,
                         metric_dir=metric_dir,
                         cell_line = cl, 
                         analysis_type = analysis_type) 
        if val is None: break

        df_file = JoinMetrics(metric_dir, cl, analysis_type)
        df_file = AddSamplingStats(df_file, row_type="sample")
        df_file["by"] = "file"

        df_group = CalculateGroupMetrics(metric_dir, cl, analysis_type)  
        df_group = AddSamplingStats(df_group, row_type="group")
        df_group["by"] = "condition"

        final = pd.concat([df_file, df_group], axis=0)
        final = final.sort_index()
        outfile=f"{metric_dir}/{cl}_by{analysis_type}.csv"
        final.to_csv(outfile)

K562
peak
K562: need to add information for 42 files
...K562_f035_s49
...K562_f025_s2
...K562_f025_s95


## Peak Subsets

In [ ]:
def CalculateMetricsPeaks(mat_dir, metric_dir, cell_line, val=50, analysis_type="tf"):
    try:
        files = _get_missing(mat_dir, metric_dir, analysis_type, cell_line)
        files = [x for x in files if "_p" not in x]
        print(f"{cell_line}: need to add information for {len(files)} files")
        if len(files) == 0: return 1
        
        peak_df = pd.read_csv(f"../06_dataquality/tfbs_universe/PeakCoverage_{cell_line}.csv.gz", index_col=0)
        
        #get the baseline dataframe, and summarize it based on type. 
        df_expected = pd.read_csv(f"{mat_dir}/{cell_line}-cellFilt.mat", index_col=0)

        if analysis_type == "peak":      df_expected_peak = df_expected.sum(axis=1).to_frame(name='sum')
        elif analysis_type == "tf":      True

        for f in files:
            cond_name = _remove_strings(f)
            df_observed = pd.read_csv(f"{mat_dir}/{f}", index_col=0)
            keep_peaks = peak_df[(peak_df[cond_name] > val) & (peak_df[cell_line] > val)].index
            print(f"...{cond_name}...{len(keep_peaks)}")
            
            df_observed_tf = df_observed.loc[keep_peaks]
            df_expected_tf = df_expected.loc[keep_peaks]

            if analysis_type == "peak":
                series_o = df_observed.sum(axis=1)
                cont_df = _make_contingency_table(df_expected_peak, series_o.to_frame(name='sum'))

            elif analysis_type == "tf":
                cont_df = _make_contingency_table(df_expected_tf, df_observed_tf)


            cont_df_with_metrics = _calc_scores(cont_df, averaging_type="macro")
            cont_df_with_metrics["type"] = analysis_type

            outfile = f"{metric_dir}/{cond_name}_metrics.csv"
            if not os.path.isfile(outfile):
                cont_df_with_metrics.to_csv(outfile)
            else: 
                print("appending results to existing file") #so you can stack tf, cluster, peak, etc. 
                cont_df_with_metrics.to_csv(outfile, mode='a', header=False)
            
        return 1
    
    except Exception as e:
        print(f"An error occurred")
        return None 
    
    

import os, re, pickle
import pandas as pd
import numpy as np
from pathlib import Path

t = "print" 
analysis_type = "tf"

for val in [100, 50]:
    data_dir = f"../06_dataquality/{t}/" #will look for mats
    Path(f"{data_dir}/metrics_{val}/").mkdir(parents=True, exist_ok=True)

    cell_lines = ["K562", "HEPG2","MCF7", "SKNSH"]
    for cl in cell_lines:
        print(cl)
        mat_dir = f"{data_dir}/mats"
        metric_dir = f"{data_dir}/metrics_{val}/"

        rc = CalculateMetricsPeaks(mat_dir=mat_dir,
                         metric_dir=metric_dir,
                         cell_line = cl, 
                         analysis_type = analysis_type,
                         val=val) 
        if rc is None: break

        df_file = JoinMetrics(metric_dir, cl, analysis_type)
        df_file = AddSamplingStats(df_file, row_type="sample")
        df_file["by"] = "file"

        df_group = CalculateGroupMetrics(metric_dir, cl, analysis_type)  
        df_group = AddSamplingStats(df_group, row_type="group")
        df_group["by"] = "condition"

        final = pd.concat([df_file, df_group], axis=0)
        final = final.sort_index()
        outfile=f"{data_dir}/metrics_{val}/{cl}_by{analysis_type}.csv"
        final.to_csv(outfile)

K562
K562: need to add information for 61 files
...K562_f08_s66...260469
